In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Section 3.3 — Simple Classification Models

**Objectives**

By the end of this section you will be able to:

- Train Logistic Regression and K-Nearest Neighbours classifiers.
- Read and interpret a confusion matrix.
- Calculate Accuracy, Precision, Recall, and F1-Score.
- Select the most relevant metric based on the business problem.

> **Cooking analogy:** Classification is like a **quality-control chef** at the
> end of a production line who examines each dish and stamps it *Pass* or *Fail*
> — or sorts it into categories such as *Starter*, *Main*, or *Dessert*. The
> model is trained on thousands of past examples until it learns the sorting
> rules automatically and can classify new dishes it has never seen.

**Classification** predicts a **category** (e.g., yes/no, tier A/B/C, fraud/not fraud).

### Logistic Regression

Despite the name, logistic regression is a **classification** algorithm. It
outputs the probability that an observation belongs to a class.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

np.random.seed(42)
n = 800

# Customer churn dataset
df_churn = pd.DataFrame({
    "Tenure_Months" : np.random.randint(1, 72, n),
    "Monthly_Spend" : np.random.normal(70, 25, n).clip(20, 200).round(2),
    "Support_Calls" : np.random.poisson(1.5, n),
    "Satisfaction"  : np.random.uniform(1, 10, n).round(1),
    "Num_Products"  : np.random.randint(1, 6, n),
})

df_churn["Churned"] = (
    (df_churn["Satisfaction"] < 4.5) |
    (df_churn["Support_Calls"] > 4)
).astype(int)

X = df_churn.drop(columns="Churned")
y = df_churn["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Dataset shape:", df_churn.shape)
print("Churn rate: {:.1%}".format(y.mean()))

In [ ]:
# Train Logistic Regression
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_sc, y_train)
y_pred_lr = log_reg.predict(X_test_sc)

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.2%}")
print(classification_report(y_test, y_pred_lr, target_names=["Stayed","Churned"]))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, name in zip(axes,
    [log_reg, KNeighborsClassifier(n_neighbors=7).fit(X_train_sc, y_train)],
    ["Logistic Regression", "K-Nearest Neighbours (k=7)"]):

    preds = model.predict(X_test_sc)
    cm    = confusion_matrix(y_test, preds)
    disp  = ConfusionMatrixDisplay(cm, display_labels=["Stayed","Churned"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}\nAccuracy: {accuracy_score(y_test, preds):.2%}")

plt.tight_layout()
plt.show()

### Understanding Classification Metrics

In [ ]:
# Manual illustration
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred_knn = KNeighborsClassifier(n_neighbors=7).fit(
    X_train_sc, y_train).predict(X_test_sc)

metrics_df = pd.DataFrame({
    "Model"     : ["Logistic Regression", "KNN (k=7)"],
    "Accuracy"  : [accuracy_score(y_test, y_pred_lr),
                   accuracy_score(y_test, y_pred_knn)],
    "Precision" : [precision_score(y_test, y_pred_lr),
                   precision_score(y_test, y_pred_knn)],
    "Recall"    : [recall_score(y_test, y_pred_lr),
                   recall_score(y_test, y_pred_knn)],
    "F1-Score"  : [f1_score(y_test, y_pred_lr),
                   f1_score(y_test, y_pred_knn)],
}).set_index("Model")

print(metrics_df.round(3))

> **Cooking analogy:** Recall is like a **food safety inspector** — it is far
> worse to *miss* a contaminated batch (false negative) than to flag a safe one
> for extra checking (false positive). In churn prediction, **Recall** (catching
> actual churners) is often more important than Precision — missing a churner is
> costlier than an unnecessary retention call.

---

### Student Task 3.3

A bank wants to classify loan applications as **Approved** or **Denied**.

1. Re-use the `df_loan` dataset from Section 3.1.
2. Train both a `LogisticRegression` and a `KNeighborsClassifier` (k=5).
3. Compare Accuracy, Precision, Recall, and F1-Score for both models.
4. Plot confusion matrices for both models side by side.
5. Which model would you recommend to the bank's risk team and why?

In [ ]:
# Your code here
# Reload df_loan from Section 3.1 if needed

---

### Evaluation Questions 3.3

1. Logistic Regression outputs a:
   a) Continuous value like revenue
   b) Probability between 0 and 1 (correct)
   c) Cluster label
   d) Feature importance score

2. **Recall** (sensitivity) measures:
   a) Of all predicted positives, how many are correct
   b) Of all actual positives, how many were correctly predicted (correct)
   c) The overall fraction of correct predictions
   d) The harmonic mean of precision and recall

3. A confusion matrix shows:
   a) Which features are most confusing for the model
   b) True and false positives and negatives (correct)
   c) The correlation between features
   d) Model training time

4. In which business scenario is **high recall** most critical?
   a) Recommending products to customers
   b) Predicting email open rates
   c) Detecting fraudulent transactions (correct)
   d) Forecasting annual revenue

5. KNN classifies a new point by:
   a) Fitting a straight decision boundary
   b) Looking at the k closest training examples and taking a majority vote (correct)
   c) Building a tree of decision rules
   d) Computing the probability using a sigmoid function

---